In [68]:
import open3d as o3d
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Palettes aligned with scripts/visualize_predgeom_contexts.py
palettes = {
    "group": np.array([[0.2, 0.6, 1.0], [1.0, 0.45, 0.2]]),
    "boundary": np.array([[0.65, 0.65, 0.65], [1.0, 0.0, 0.0]]),
    "range": np.array([[0.2, 0.7, 0.3], [0.95, 0.8, 0.1], [0.6, 0.2, 0.9]]),
    "mode": np.array([
        [0.6, 0.6, 0.6],
        [0.1, 0.7, 0.95],
        [0.2, 0.6, 0.2],
        [0.9, 0.4, 0.1],
        [0.8, 0.2, 0.7],
        [0.95, 0.9, 0.2],
    ]),
}

def get_colored_cloud(points, values, palette):
    colors = np.zeros((points.shape[0], 3), dtype=np.float64)
    unknown_color = np.array([1.0, 0.0, 1.0], dtype=np.float64)
    colors[:] = unknown_color
    for idx, color in enumerate(palette):
        colors[values == idx] = color

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    return pcd

def prepare_view(points, values, z_scale=3.0, sample_step=1):
    vis_points = points.copy()
    vis_points[:, 2] *= float(z_scale)
    step = max(1, int(sample_step))
    if step > 1:
        vis_points = vis_points[::step]
        values = values[::step]
    return vis_points, values

def _plotly_scene_keys(fig):
    layout = fig.to_dict().get("layout", {})
    keys = [k for k in layout.keys() if str(k).startswith("scene")]
    return keys or ["scene"]

def _figure_xyz_bounds(fig):
    xs, ys, zs = [], [], []
    for tr in fig.data:
        if getattr(tr, "type", None) != "scatter3d":
            continue
        x = np.asarray(getattr(tr, "x", []), dtype=np.float64)
        y = np.asarray(getattr(tr, "y", []), dtype=np.float64)
        z = np.asarray(getattr(tr, "z", []), dtype=np.float64)
        if x.size:
            xs.append(x[np.isfinite(x)])
        if y.size:
            ys.append(y[np.isfinite(y)])
        if z.size:
            zs.append(z[np.isfinite(z)])
    if not xs or not ys or not zs:
        return None
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    z = np.concatenate(zs)
    if x.size == 0 or y.size == 0 or z.size == 0:
        return None
    return (x.min(), x.max(), y.min(), y.max(), z.min(), z.max())

def _apply_plotly_view(fig, lock_origin=True, eye=(1.25, 1.25, 0.8), axis_half_extent=None, zoom_factor=None):
    h = None
    cx = cy = cz = 0.0
    if axis_half_extent is not None:
        h = float(axis_half_extent)
    elif zoom_factor is not None:
        b = _figure_xyz_bounds(fig)
        if b is not None:
            xmin, xmax, ymin, ymax, zmin, zmax = b
            cx = 0.5 * (xmin + xmax)
            cy = 0.5 * (ymin + ymax)
            cz = 0.5 * (zmin + zmax)
            span = max(xmax - xmin, ymax - ymin, zmax - zmin)
            zf = max(1e-6, float(zoom_factor))
            h = max(1e-9, span / zf)

    for scene_key in _plotly_scene_keys(fig):
        scene_update = {"aspectmode": "data"}
        if lock_origin:
            scene_update["camera"] = {
                "up": {"x": 0.0, "y": 0.0, "z": 1.0},
                "center": {"x": 0.0, "y": 0.0, "z": 0.0},
                "eye": {"x": float(eye[0]), "y": float(eye[1]), "z": float(eye[2])},
            }
        if h is not None:
            scene_update["xaxis"] = {"range": [cx - h, cx + h], "autorange": False}
            scene_update["yaxis"] = {"range": [cy - h, cy + h], "autorange": False}
            scene_update["zaxis"] = {"range": [cz - h, cz + h], "autorange": False}
        fig.update_layout(**{scene_key: scene_update})

def render_plotly_cloud(
    pcd,
    width=1200,
    height=800,
    lock_origin=True,
    eye=(1.25, 1.25, 0.8),
    axis_half_extent=None,
    zoom_factor=None,
    attr_name=None,
    values=None,
    palette=None,
):
    fig_raw = o3d.visualization.draw_plotly([pcd], width=int(width), height=int(height))
    fig = fig_raw if isinstance(fig_raw, go.Figure) else go.Figure(fig_raw)
    _apply_plotly_view(
        fig,
        lock_origin=lock_origin,
        eye=eye,
        axis_half_extent=axis_half_extent,
        zoom_factor=zoom_factor,
    )

    if attr_name is not None and values is not None and palette is not None:
        label_names = {
            "group": ["group 0", "group 1"],
            "boundary": ["boundary 0", "boundary 1"],
            "range": ["range 0", "range 1", "range 2"],
            "mode": [
                "mode 0 (zero)",
                "mode 1 (same-last)",
                "mode 2 (same-lin2)",
                "mode 3 (cross-up)",
                "mode 4 (cross-down)",
                "mode 5 (hist1)",
            ],
        }
        names = label_names.get(attr_name, [f"{attr_name} {i}" for i in range(len(palette))])
        present_labels = sorted(int(v) for v in np.unique(values) if 0 <= int(v) < len(palette))
        for i in present_labels:
            rgb = tuple(int(round(255 * c)) for c in palette[i])
            fig.add_trace(
                go.Scatter3d(
                    x=[None],
                    y=[None],
                    z=[None],
                    mode="markers",
                    marker={"size": 8, "color": f"rgb{rgb}"},
                    name=names[i] if i < len(names) else f"{attr_name} {i}",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )

    fig.update_layout(legend={"title": {"text": str(attr_name) if attr_name else "label"}})
    return fig

def render_open3d_cloud(pcd, lock_origin=True, lookat=(0.0, 0.0, 0.0), front=(1.25, 1.25, 0.8), up=(0.0, 0.0, 1.0), zoom=0.7, window_name="PredGeom Open3D"):
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name=window_name, width=1280, height=900)
    vis.add_geometry(pcd)
    opt = vis.get_render_option()
    opt.point_size = 2.0

    if lock_origin:
        ctr = vis.get_view_control()
        ctr.set_lookat(np.array(lookat, dtype=np.float64))
        ctr.set_front(np.array(front, dtype=np.float64))
        ctr.set_up(np.array(up, dtype=np.float64))
        ctr.set_zoom(float(zoom))

    vis.run()
    vis.destroy_window()


In [69]:
# SET YOUR PATHS HERE
ply_path = "/home/swnh/gpcc/build/sandbox/input.ply"
csv_path = "/home/swnh/gpcc/build/sandbox/context_dump.csv"

required_cols = [
    "src_idx",
    "ctx_group",
    "mode_boundary",
    "mode_range_class",
    "mode",
]

df = pd.read_csv(csv_path)
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

pcd_orig = o3d.io.read_point_cloud(ply_path)
points = np.asarray(pcd_orig.points)
if points.shape[0] == 0:
    raise ValueError("PLY has zero points")

src_idx = df["src_idx"].to_numpy(dtype=np.int64)
if src_idx.max() >= len(points) or src_idx.min() < 0:
    raise ValueError("src_idx out of bounds for this PLY")

group_vals = np.zeros(len(points), dtype=np.int32)
boundary_vals = np.zeros(len(points), dtype=np.int32)
range_vals = np.zeros(len(points), dtype=np.int32)
mode_vals = np.zeros(len(points), dtype=np.int32)

group_vals[src_idx] = df["ctx_group"].to_numpy(dtype=np.int32)
boundary_vals[src_idx] = df["mode_boundary"].to_numpy(dtype=np.int32)
range_vals[src_idx] = df["mode_range_class"].to_numpy(dtype=np.int32)
mode_vals[src_idx] = df["mode"].to_numpy(dtype=np.int32)

print(f"Loaded {len(points)} points, {len(df)} CSV rows")
print(f"Context groups in dump: {sorted(df['ctx_group'].dropna().astype(int).unique().tolist())}")
print(f"Unique src_idx mapped: {len(np.unique(src_idx))}")


Loaded 34688 points, 34688 CSV rows
Context groups in dump: [0, 1]
Unique src_idx mapped: 34688


In [70]:
def _require_loaded_data():
    needed = ["points", "group_vals", "boundary_vals", "range_vals", "mode_vals"]
    missing = [name for name in needed if name not in globals()]
    if missing:
        raise RuntimeError(
            "Missing loaded variables: " + ", ".join(missing)
            + ". Run the data-loading cell before calling show_attr()."
        )

def _attr_map():
    _require_loaded_data()
    return {
        "group": group_vals,
        "boundary": boundary_vals,
        "range": range_vals,
        "mode": mode_vals,
    }

def show_attr(
    attr="mode",
    z_scale=3.0,
    sample_step=1,
    backend="plotly",
    width=1200,
    height=800,
    lock_origin=True,
    eye=(1.25, 1.25, 0.8),
    axis_half_extent=None,
    zoom_factor=None,
    auto_show=True,
    scroll_zoom=True,
    open3d_front=(1.25, 1.25, 0.8),
    open3d_up=(0.0, 0.0, 1.0),
    open3d_zoom=0.7,
):
    attrs = _attr_map()
    if attr not in attrs:
        raise ValueError(f"attr must be one of {list(attrs.keys())}")
    if backend not in ("plotly", "open3d"):
        raise ValueError("backend must be 'plotly' or 'open3d'")

    values = attrs[attr]
    vis_points, vis_values = prepare_view(points, values, z_scale=z_scale, sample_step=sample_step)
    pcd = get_colored_cloud(vis_points, vis_values, palettes[attr])

    u, c = np.unique(vis_values, return_counts=True)
    stats = {int(k): int(v) for k, v in zip(u, c)}
    print(f"Showing '{attr}' with backend='{backend}' | z_scale={z_scale} | sample_step={max(1, int(sample_step))} | points={len(vis_points)}")
    print(f"Label counts: {stats}")

    if backend == "plotly":
        fig = render_plotly_cloud(
            pcd,
            width=width,
            height=height,
            lock_origin=lock_origin,
            eye=eye,
            axis_half_extent=axis_half_extent,
            zoom_factor=zoom_factor,
            attr_name=attr,
            values=vis_values,
            palette=palettes[attr],
        )
        if auto_show:
            fig.show(config={"scrollZoom": bool(scroll_zoom), "displaylogo": False, "responsive": True})
        return fig

    render_open3d_cloud(
        pcd,
        lock_origin=lock_origin,
        lookat=(0.0, 0.0, 0.0),
        front=open3d_front,
        up=open3d_up,
        zoom=open3d_zoom,
        window_name=f"PredGeom {attr}",
    )
    return None


In [75]:
fig = show_attr(
    attr="mode",
    backend="plotly",
    z_scale=3.0,
    sample_step=1,
    lock_origin=True,
    zoom_factor=5.0,
    auto_show=False,
)
fig.update_traces(marker=dict(size=4))
fig.show(config={"scrollZoom": True, "displaylogo": False, "responsive": True})
fig


Showing 'mode' with backend='plotly' | z_scale=3.0 | sample_step=1 | points=34688
Label counts: {0: 567, 1: 7187, 2: 22054, 3: 2218, 4: 706, 5: 1956}
